# GLADA: Comprehensive Eval — Round 2

Fills in missing evaluations from Round 1. Smart per-(model, dataset) skipping.
Merges into existing results.json.

In [ ]:
import os, sys, json, time, datetime, tarfile, zipfile
from pathlib import Path
from dataclasses import dataclass, field, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
    if 'a100' in gpu_name.lower() or 'h100' in gpu_name.lower():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

AMP_DTYPE = torch.bfloat16 if device.type == 'cuda' and (
    'a100' in torch.cuda.get_device_name(0).lower() or
    'h100' in torch.cuda.get_device_name(0).lower()
) else torch.float16
print(f'AMP dtype: {AMP_DTYPE}')

RESULTS_DIR = '/content/drive/MyDrive/Jakob/Experiments/comprehensive_eval'
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS_FILE = os.path.join(RESULTS_DIR, 'results.json')

if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        all_results = json.load(f)
    print(f'Loaded {len(all_results)} existing model results.')
    for name, data in all_results.items():
        ds = [k for k, v in data.get('datasets', {}).items()
              if isinstance(v, dict) and v.get('eer', -1) >= 0]
        print(f'  {name}: {ds}')
else:
    all_results = {}
    print('No existing results found.')

Mounted at /content/drive
Device: cuda
GPU: Tesla T4 (15.6 GB)
AMP dtype: torch.float16
Loaded 16 existing model results.
  full_glada: ['2019_LA', 'ITW']
  no_glada: ['2019_LA', 'ITW']
  no_vib: ['2019_LA', 'ITW']
  no_speaker: ['2019_LA', 'ITW']
  no_cka: ['2019_LA', 'ITW']
  no_rawboost: ['2019_LA', 'ITW']
  no_hier: ['2019_LA', 'ITW']
  global_da_only: ['2019_LA', 'ITW']
  hierarchical_only: ['2019_LA', 'ITW']
  no_da: ['2019_LA', 'ITW']
  backbone_only: ['2019_LA', 'ITW']
  bb_wavlm-large_s42: ['2019_LA', 'ITW']
  bb_wavlm-large_s123: ['2019_LA', 'ITW']
  bb_xlsr-300m_s42: ['2019_LA', 'ITW']
  bb_data2vec-large: ['2019_LA', 'ITW']
  bb_wav2vec2-robust: ['ITW']


In [ ]:
GLADA_PKG_PATH = '/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code'
if GLADA_PKG_PATH not in sys.path:
    sys.path.insert(0, GLADA_PKG_PATH)

from glada.config import GLADAConfig, BACKBONE_REGISTRY, set_seed
from glada.models.glada_model import GLADAModel
from glada.datasets.audio import load_audio
from glada.datasets.asvspoof import ASVspoofEvalDataset
from glada.datasets.itw import InTheWildDataset
from glada.evaluation.metrics import compute_eer, compute_eer_with_ci
from transformers import AutoModel

set_seed(42)
print('Package loaded.')

Package loaded.


## Extract Datasets

In [ ]:
LOCAL_ROOT = '/content/data'
os.makedirs(LOCAL_ROOT, exist_ok=True)
DRIVE_BASE = '/content/drive/MyDrive/Jakob/Datasets'


def existing_path(*candidates):
    """Return first existing path from a list of candidates."""
    for p in candidates:
        if p and os.path.exists(p):
            return p
    return None


def extract_tar(tar_path, extract_to, name):
    if not tar_path or not os.path.exists(tar_path):
        print(f'  [{name}] Archive not found: {tar_path}')
        return False
    os.makedirs(extract_to, exist_ok=True)
    if os.listdir(extract_to):
        print(f'  [{name}] Already extracted')
        return True
    print(f'  [{name}] Extracting {tar_path}...')
    mode = 'r:gz' if tar_path.endswith('.gz') else 'r:*'
    try:
        with tarfile.open(tar_path, mode) as tar:
            tar.extractall(path=extract_to)
    except (tarfile.ReadError, EOFError, OSError) as e:
        print(f'  [{name}] FAILED (likely corrupted archive): {e}')
        print(f"  [{name}] SKIP FOR NOW. Remove and re-download archive on Drive, then rerun this cell.")
        return False
    print(f'  [{name}] Done')
    return True


# === ASVspoof 2019 LA ===
la_2019_archive = existing_path(
    os.path.join(DRIVE_BASE, 'LA.tar'),
    os.path.join(DRIVE_BASE, 'ASVspoof2019_LA.tar.gz'),
)
extract_tar(la_2019_archive, os.path.join(LOCAL_ROOT, 'asvspoof'), 'asvspoof19')

# === ITW ===
extract_tar(os.path.join(DRIVE_BASE, 'release_in_the_wild.tar'),
            os.path.join(LOCAL_ROOT, 'itw'), 'itw')

# === ASVspoof 2021 LA ===
extract_tar(os.path.join(DRIVE_BASE, 'ASVspoof2021_LA_eval.tar.gz'),
            os.path.join(LOCAL_ROOT, 'asvspoof21_la'), 'asvspoof21la')

# === ASVspoof 2021 DF (already extracted on Drive, extract inner archives) ===
DF_DRIVE = os.path.join(DRIVE_BASE, 'ASVspoof2021_DF')
DF_LOCAL = os.path.join(LOCAL_ROOT, 'asvspoof21_df')
if os.path.isdir(DF_DRIVE):
    os.makedirs(DF_LOCAL, exist_ok=True)

    # Extract keys archive if present
    df_keys_tar = existing_path(
        os.path.join(DF_DRIVE, 'DF-keys-full.tar'),
        os.path.join(DF_DRIVE, 'keys', 'DF-keys-full.tar'),
    )
    if df_keys_tar:
        print(f'  [2021_DF] Extracting keys from {os.path.basename(df_keys_tar)}...')
        try:
            with tarfile.open(df_keys_tar, 'r:*') as tar:
                tar.extractall(path=DF_LOCAL)
        except (tarfile.ReadError, EOFError, OSError) as e:
            print(f'  [2021_DF] Keys archive corrupted/unreadable: {e}')

    # Extract audio part archives
    df_flac_check = os.path.join(DF_LOCAL, 'flac')
    if not os.path.isdir(df_flac_check):
        part_archives = [
            part for part in sorted(os.listdir(DF_DRIVE))
            if part.startswith('ASVspoof2021_DF_eval') and part.endswith('.tar.gz')
        ]
        if not part_archives:
            print('  [2021_DF] No part archives found in DF root; listing directory for debugging:')
            print('   ', sorted(os.listdir(DF_DRIVE))[:30])

        failed_parts = []
        for part in part_archives:
            print(f'  [2021_DF] Extracting {part}...')
            try:
                with tarfile.open(os.path.join(DF_DRIVE, part), 'r:gz') as tar:
                    tar.extractall(path=DF_LOCAL)
            except (tarfile.ReadError, EOFError, OSError) as e:
                failed_parts.append((part, str(e)))
                print(f'  [2021_DF] FAILED {part}: {e}')

        if failed_parts:
            print('  [2021_DF] Some archives are corrupted. Continuing with available datasets for now.')
            print('  [2021_DF] Re-download these parts on Drive, then rerun extraction:')
            for part, err in failed_parts:
                print(f'    - {part}: {err}')
        else:
            print('  [2021_DF] Audio extraction done.')
    else:
        print('  [2021_DF] Already extracted')
else:
    print(f'  [2021_DF] NOT FOUND on Drive at {DF_DRIVE}')

# === WaveFake (zip) ===
WF_LOCAL = os.path.join(LOCAL_ROOT, 'wavefake')
WF_ZIP = os.path.join(DRIVE_BASE, 'wavefake.zip')
os.makedirs(WF_LOCAL, exist_ok=True)
if os.listdir(WF_LOCAL):
    print('  [wavefake] Already extracted')
elif os.path.exists(WF_ZIP):
    print(f'  [wavefake] Extracting {WF_ZIP}...')
    with zipfile.ZipFile(WF_ZIP, 'r') as zf:
        zf.extractall(WF_LOCAL)
    print('  [wavefake] Done')
else:
    wf_drive = os.path.join(DRIVE_BASE, 'wavefake')
    if os.path.isdir(wf_drive):
        os.rmdir(WF_LOCAL)
        os.symlink(wf_drive, WF_LOCAL)
        print('  [wavefake] Symlinked from Drive')
    else:
        print('  [wavefake] NOT FOUND')

print('\n=== Extraction complete ===')


  [asvspoof19] Extracting /content/drive/MyDrive/Jakob/Datasets/LA.tar...


/tmp/ipykernel_371/139868742.py:26: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_to)


  [asvspoof19] Done
  [itw] Extracting /content/drive/MyDrive/Jakob/Datasets/release_in_the_wild.tar...
  [itw] Done
  [asvspoof21la] Extracting /content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_LA_eval.tar.gz...
  [asvspoof21la] Done
  [2021_DF] Extracting keys from DF-keys-full.tar...


/tmp/ipykernel_371/139868742.py:65: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=DF_LOCAL)


  [2021_DF] Extracting ASVspoof2021_DF_eval_full.tar.gz...


/tmp/ipykernel_371/139868742.py:85: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=DF_LOCAL)


  [2021_DF] FAILED ASVspoof2021_DF_eval_full.tar.gz: Compressed file ended before the end-of-stream marker was reached
  [2021_DF] Extracting ASVspoof2021_DF_eval_part00.tar.gz...
  [2021_DF] Extracting ASVspoof2021_DF_eval_part01.tar.gz...
  [2021_DF] Extracting ASVspoof2021_DF_eval_part02.tar.gz...
  [2021_DF] Extracting ASVspoof2021_DF_eval_part03.tar.gz...
  [2021_DF] Some archives are corrupted. Continuing with available datasets for now.
  [2021_DF] Re-download these parts on Drive, then rerun extraction:
    - ASVspoof2021_DF_eval_full.tar.gz: Compressed file ended before the end-of-stream marker was reached
  [wavefake] Extracting /content/drive/MyDrive/Jakob/Datasets/wavefake.zip...
  [wavefake] Done

=== Extraction complete ===


## Verify Paths (run this, check output, then continue)

In [ ]:
# Show actual directory structure so we can fix dataloader paths if needed
import os

print('=== Checking all dataset locations ===\n')


def show_tree(path, depth=2, prefix=''):
    if not os.path.isdir(path):
        print(f'{prefix}{path} -- NOT FOUND')
        return
    items = sorted(os.listdir(path))
    for item in items[:15]:
        full = os.path.join(path, item)
        is_dir = os.path.isdir(full)
        print(f'{prefix}{item}{"/" if is_dir else ""}')
        if is_dir and depth > 1:
            show_tree(full, depth - 1, prefix + '  ')


for name, path in [
    ('asvspoof (2019 LA)', '/content/data/asvspoof'),
    ('itw', '/content/data/itw'),
    ('asvspoof21_la', '/content/data/asvspoof21_la'),
    ('asvspoof21_df', '/content/data/asvspoof21_df'),
    ('wavefake', '/content/data/wavefake'),
]:
    print(f'\n--- {name} ---')
    show_tree(path)

# Specific file checks the dataloaders need
print('\n=== Critical file checks ===')
critical = {
    '2019_LA proto':  '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt',
    '2019_LA audio':  '/content/data/asvspoof/LA/ASVspoof2019_LA_eval/flac',
    'ITW meta':       '/content/data/itw/release_in_the_wild/meta.csv',
    '2021_LA proto':  '/content/data/asvspoof21_la/ASVspoof2021_LA_eval/ASVspoof2021.LA.cm.eval.trl.txt',
    '2021_LA audio':  '/content/data/asvspoof21_la/ASVspoof2021_LA_eval/flac',
    '2021_DF keys':   '/content/data/asvspoof21_df/keys/DF/CM/trial_metadata.txt',
    '2021_DF audio':  '/content/data/asvspoof21_df/flac',
    'WaveFake root':  '/content/data/wavefake/generated_audio',
}
for label, path in critical.items():
    print(f'  {label:20s}: {"OK" if os.path.exists(path) else "MISSING"} -> {path}')

if not os.path.exists(critical['2019_LA audio']):
    print('\n[2019_LA debug] Eval audio is missing. Check Drive for alternate archives:')
    print('  !ls /content/drive/MyDrive/Jakob/Datasets | grep -i -E "2019|LA|asvspoof"')
if not os.path.exists(critical['2021_DF keys']) or not os.path.exists(critical['2021_DF audio']):
    print('\n[2021_DF debug] Missing keys/audio. Verify extracted layout:')
    print('  !ls -lah /content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_DF | head -30')
    print('  !find /content/data/asvspoof21_df -maxdepth 3 -type d | head -30')


=== Checking all dataset locations ===


--- asvspoof (2019 LA) ---
LA/
  ASVspoof2019_LA_asv_protocols/
  ASVspoof2019_LA_asv_scores/
  ASVspoof2019_LA_cm_protocols/
  ASVspoof2019_LA_dev/
  ASVspoof2019_LA_eval/
  ASVspoof2019_LA_train/
  README.LA.txt

--- itw ---
release_in_the_wild/
  0.wav
  1.wav
  10.wav
  100.wav
  1000.wav
  10000.wav
  10001.wav
  10002.wav
  10003.wav
  10004.wav
  10005.wav
  10006.wav
  10007.wav
  10008.wav
  10009.wav

--- asvspoof21_la ---
ASVspoof2021_LA_eval/
  ASVspoof2021.LA.cm.eval.trl.txt
  LICENSE.txt
  README.LA.txt
  flac/

--- asvspoof21_df ---
ASVspoof2021_DF_eval/
  ASVspoof2021.DF.cm.eval.trl.txt
  LICENSE.DF.txt
  README.DF.txt
  flac/
keys/
  DF/

--- wavefake ---
generated_audio/
  common_voices_prompts_from_conformer_fastspeech2_pwg_ljspeech/
  jsut_multi_band_melgan/
  jsut_parallel_wavegan/
  ljspeech_full_band_melgan/
  ljspeech_hifiGAN/
  ljspeech_melgan/
  ljspeech_melgan_large/
  ljspeech_multi_band_melgan/
  ljspeech_parallel_wa

## Dataset Classes

In [ ]:
class ASVspoof2021Dataset(Dataset):
    """ASVspoof 2021 LA or DF evaluation set (Yamagishi et al., 2021).

    Handles two different protocol formats:
      - LA: ASVspoof2021.LA.cm.eval.trl.txt (space-separated, key in col 1)
      - DF: trial_metadata.txt (space-separated, key in col 1, 'bonafide' in line)
    """
    def __init__(self, keys_path, audio_dir, max_len=64000):
        self.audio_dir = Path(audio_dir)
        self.max_len = max_len
        self.samples = []
        with open(keys_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 2:
                    continue
                filename = parts[1]
                label = 1 if 'bonafide' in line else 0
                self.samples.append({'filename': filename, 'label': label})
        n_bona = sum(1 for s in self.samples if s['label'] == 1)
        print(f'  ASVspoof 2021: {len(self.samples)} samples '
              f'({n_bona} bonafide, {len(self.samples)-n_bona} spoof)')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        audio_path = self.audio_dir / f"{s['filename']}.flac"
        if not audio_path.exists():
            audio_path = self.audio_dir / f"{s['filename']}.wav"
        wav = load_audio(str(audio_path), self.max_len)
        mask = torch.ones(self.max_len, dtype=torch.long)
        return {'input_values': wav, 'attention_mask': mask,
                'task_label': torch.tensor(s['label'], dtype=torch.long)}


class WaveFakeDataset(Dataset):
    """WaveFake benchmark (Frank & Schonherr, 2021).

    Audio is under generated_audio/ with one subdir per vocoder (all spoof).
    Bonafide LJSpeech samples may or may not be present.
    If no bonafide found, this dataset cannot compute EER.
    """
    def __init__(self, root_dir, max_len=64000):
        self.max_len = max_len
        self.samples = []
        root = Path(root_dir)
        # Walk all subdirectories recursively
        for subdir in sorted(root.rglob('*')):
            if not subdir.is_dir():
                continue
            # Bonafide: anything with 'ljspeech-1.1' in path (case-insensitive)
            is_bonafide = 'ljspeech-1.1' in str(subdir).lower()
            label = 1 if is_bonafide else 0
            for wav_path in subdir.glob('*.wav'):
                self.samples.append({'path': str(wav_path), 'label': label})
        # Deduplicate (rglob can hit same files)
        seen = set()
        unique = []
        for s in self.samples:
            if s['path'] not in seen:
                seen.add(s['path'])
                unique.append(s)
        self.samples = unique
        n_bona = sum(1 for s in self.samples if s['label'] == 1)
        n_spoof = len(self.samples) - n_bona
        print(f'  WaveFake: {len(self.samples)} samples '
              f'({n_bona} bonafide, {n_spoof} spoof)')
        if n_bona == 0:
            print('  WARNING: No bonafide samples found! EER will be undefined.')
            print('  WaveFake only distributes generated audio.')
            print('  Bonafide LJSpeech-1.1 must be downloaded separately.')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        wav = load_audio(s['path'], self.max_len)
        mask = torch.ones(self.max_len, dtype=torch.long)
        return {'input_values': wav, 'attention_mask': mask,
                'task_label': torch.tensor(s['label'], dtype=torch.long)}


print('Dataset classes defined.')

Dataset classes defined.


## Model Loading

In [ ]:
def load_glada_model(checkpoint_path, backbone='wavlm-base-plus'):
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    saved_config = ckpt.get('config', {})
    config = GLADAConfig(backbone=saved_config.get('backbone', backbone))
    for k, v in saved_config.items():
        if hasattr(config, k):
            setattr(config, k, v)
    config.__post_init__()
    model = GLADAModel(config)
    model.load_state_dict(ckpt['model'])
    model.eval()
    print(f'  GLADA: {config.backbone}, ep={ckpt.get("epoch","?")}, EER={ckpt.get("best_eer","?")}')
    return model, asdict(config)


class _LegacyGRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()
    @staticmethod
    def backward(ctx, grad):
        return -ctx.alpha * grad, None


class LegacyDomainAdaptationModel(nn.Module):
    def __init__(self, model_name, hidden_size, has_domain_disc=True,
                 has_bottleneck=False, bn_dim=16):
        super().__init__()
        self.wavlm = AutoModel.from_pretrained(model_name)
        self.wavlm.feature_extractor._freeze_parameters()
        for p in self.wavlm.parameters():
            p.requires_grad = False
        self.has_bottleneck = has_bottleneck
        if has_bottleneck:
            self.bottleneck = nn.Sequential(nn.Linear(hidden_size, bn_dim), nn.ReLU())
            cls_input_dim = bn_dim
        else:
            self.bottleneck = None
            cls_input_dim = hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(cls_input_dim, 256), nn.LayerNorm(256),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 2))
        self.domain_disc = nn.Sequential(
            nn.Linear(cls_input_dim, 256), nn.LayerNorm(256),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 2)) if has_domain_disc else None

    def forward(self, input_values, attention_mask=None):
        with torch.no_grad():
            features = self.wavlm(
                input_values, attention_mask=attention_mask
            ).last_hidden_state.mean(1)
        if self.has_bottleneck:
            features = self.bottleneck(features)
        return {'spoof_logits': self.classifier(features), 'features': features}


LEGACY_BACKBONE_MAP = {
    'wavlm_base_plus':  ('microsoft/wavlm-base-plus', 768),
    'wavlm_large':      ('microsoft/wavlm-large', 1024),
    'xlsr_300m':         ('facebook/wav2vec2-xls-r-300m', 1024),
    'wav2vec2_base':     ('facebook/wav2vec2-base-960h', 768),
    'wav2vec2_large':    ('facebook/wav2vec2-large-960h', 1024),
}


def load_legacy_model(checkpoint_path, backbone='wavlm_base_plus'):
    model_name, hidden = LEGACY_BACKBONE_MAP[backbone]
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    if isinstance(ckpt, dict) and 'model' in ckpt:
        state_dict = ckpt['model']
        info = {k: v for k, v in ckpt.items() if k != 'model'}
    else:
        state_dict = ckpt
        info = {}
    has_disc = any('domain_disc' in k for k in state_dict.keys())
    has_bn = any('bottleneck' in k for k in state_dict.keys())
    bn_dim = 16
    if has_bn:
        for k, v in state_dict.items():
            if 'bottleneck.0.weight' in k:
                bn_dim = v.shape[0]
                break
    model = LegacyDomainAdaptationModel(model_name, hidden, has_disc, has_bn, bn_dim)
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    print(f'  Legacy: {backbone}, bn={has_bn}({bn_dim}), ep={info.get("epoch","?")}')
    return model, info


print('Model loading ready.')

Model loading ready.


## Evaluator

In [ ]:
@torch.no_grad()
def evaluate_model_on_dataset(model, loader, device, dataset_name='dataset',
                              amp_dtype=torch.float16, n_bootstrap=1000):
    model.eval()
    all_scores, all_labels = [], []
    for batch in tqdm(loader, desc=f'Eval {dataset_name}', leave=False):
        inp = batch['input_values'].to(device, non_blocking=True)
        mask = batch['attention_mask'].to(device, non_blocking=True)
        with autocast('cuda', enabled=(device.type == 'cuda'), dtype=amp_dtype):
            out = model(inp, attention_mask=mask)
        probs = F.softmax(out['spoof_logits'], dim=1)[:, 1]
        all_scores.extend(probs.float().cpu().numpy())
        all_labels.extend(batch['task_label'].numpy())
    labels = np.array(all_labels)
    scores = np.array(all_scores)
    metrics = compute_eer_with_ci(labels, scores, n_bootstrap=n_bootstrap)
    metrics['n_samples'] = len(labels)
    metrics['name'] = dataset_name
    return metrics


def evaluate_missing_only(model, eval_loaders, existing_ds, device,
                          model_name='model', amp_dtype=torch.float16):
    results = {}
    for ds_name, loader in eval_loaders.items():
        if ds_name in existing_ds:
            print(f'  {ds_name:<15s}: SKIP (already done)')
            continue
        try:
            r = evaluate_model_on_dataset(model, loader, device, ds_name, amp_dtype)
            results[ds_name] = r
            print(f'  {ds_name:<15s}: EER={r["eer"]:.2f}% '
                  f'[{r["ci_low"]:.2f}, {r["ci_high"]:.2f}] (n={r["n_samples"]:,})')
        except Exception as e:
            print(f'  {ds_name:<15s}: ERROR - {e}')
            results[ds_name] = {'eer': -1, 'error': str(e)}
    return results


print('Evaluator ready.')

Evaluator ready.


## Checkpoint Registry

In [ ]:
DRIVE_EXP = '/content/drive/MyDrive/Jakob/Experiments'


def find_checkpoint(base_dir, ckpt_name):
    if not os.path.isdir(base_dir):
        return None
    direct = os.path.join(base_dir, ckpt_name)
    if os.path.exists(direct):
        return direct
    found = []
    for sub in sorted(os.listdir(base_dir)):
        sub_path = os.path.join(base_dir, sub)
        if os.path.isdir(sub_path):
            candidate = os.path.join(sub_path, ckpt_name)
            if os.path.exists(candidate):
                found.append(candidate)
    return found[-1] if found else None


GROUP_A_SPEC = [
    ('full_glada',        'ablation_full_glada',           'best_model.pt', 'wavlm-base-plus'),
    ('no_glada',          'ablation_no_glada',             'best_model.pt', 'wavlm-base-plus'),
    ('no_vib',            'ablation_no_vib',               'best_model.pt', 'wavlm-base-plus'),
    ('no_speaker',        'ablation_no_speaker',           'best_model.pt', 'wavlm-base-plus'),
    ('no_cka',            'ablation_no_cka',               'best_model.pt', 'wavlm-base-plus'),
    ('no_rawboost',       'ablation_no_rawboost',          'best_model.pt', 'wavlm-base-plus'),
    ('no_hier',           'ablation_no_hier',              'best_model.pt', 'wavlm-base-plus'),
    ('global_da_only',    'ablation_global_da_only',       'best_model.pt', 'wavlm-base-plus'),
    ('hierarchical_only', 'ablation_hierarchical_only',    'best_model.pt', 'wavlm-base-plus'),
    ('no_da',             'ablation_no_da',                'best_model.pt', 'wavlm-base-plus'),
    ('backbone_only',     'ablation_backbone_only',        'best_model.pt', 'wavlm-base-plus'),
]

GROUP_B_SPEC = [
    ('bb_wavlm-large_s42',  'ablation_full_wavlm-large',       'best_model.pt', 'wavlm-large'),
    ('bb_wavlm-large_s123', 'ablation_full_wavlm-large_s123',  'best_model.pt', 'wavlm-large'),
    ('bb_xlsr-300m_s42',    'ablation_full_xlsr-300m_s42',      'best_model.pt', 'xlsr-300m'),
    ('bb_data2vec-large',   'ablation_full_data2vec-large',     'best_model.pt', 'data2vec-large'),
    ('bb_wav2vec2-robust',  'ablation_full_wav2vec2-robust',    'best_model.pt', 'wav2vec2-robust'),
]

GROUP_C_SPEC = [
    ('sem1_wbp_no_adapt', 'wavlm_base_plus__no_adapt__seed42',  'best_model.pt', 'wavlm_base_plus'),
    ('sem1_wbp_dann_ib',  'wavlm_base_plus__dann_ib__seed42',   'best_model.pt', 'wavlm_base_plus'),
    ('sem1_wbp_caan',     'wavlm_base_plus__caan__seed42',      'best_model.pt', 'wavlm_base_plus'),
    ('sem1_wl_dann_ib',   'wavlm_large__dann_ib__seed42',       'best_model.pt', 'wavlm_large'),
    ('sem1_wl_caan',      'wavlm_large__caan__seed42',          'best_model.pt', 'wavlm_large'),
    ('sem1_xlsr_dann_ib', 'xlsr_300m__dann_ib__seed42',         'best_model.pt', 'xlsr_300m'),
]

ALL_MODELS = []
print('Resolving checkpoints...\n')
for group_name, specs, loader_type in [
    ('Group A: Ablation', GROUP_A_SPEC, 'glada'),
    ('Group B: Backbones', GROUP_B_SPEC, 'glada'),
    ('Group C: Semester 1', GROUP_C_SPEC, 'legacy'),
]:
    print(f'--- {group_name} ---')
    for display_name, exp_dir, ckpt_name, backbone in specs:
        candidates = [os.path.join(DRIVE_EXP, exp_dir)]
        if loader_type == 'legacy':
            candidates.append(os.path.join(DRIVE_EXP, 'exp_C_comprehensive', exp_dir))
        resolved = None
        for base in candidates:
            resolved = find_checkpoint(base, ckpt_name)
            if resolved:
                break
        if resolved:
            ALL_MODELS.append((display_name, resolved, loader_type, backbone))
            print(f'  {display_name:<30s} FOUND')
        else:
            ALL_MODELS.append((display_name, 'NOT_FOUND', loader_type, backbone))
            print(f'  {display_name:<30s} NOT FOUND')
    print()

n_found = sum(1 for _, p, _, _ in ALL_MODELS if p != 'NOT_FOUND')
print(f'Found: {n_found}/{len(ALL_MODELS)}')

Resolving checkpoints...

--- Group A: Ablation ---
  full_glada                     FOUND
  no_glada                       FOUND
  no_vib                         FOUND
  no_speaker                     FOUND
  no_cka                         FOUND
  no_rawboost                    FOUND
  no_hier                        FOUND
  global_da_only                 FOUND
  hierarchical_only              FOUND
  no_da                          FOUND
  backbone_only                  FOUND

--- Group B: Backbones ---
  bb_wavlm-large_s42             FOUND
  bb_wavlm-large_s123            FOUND
  bb_xlsr-300m_s42               FOUND
  bb_data2vec-large              FOUND
  bb_wav2vec2-robust             FOUND

--- Group C: Semester 1 ---
  sem1_wbp_no_adapt              FOUND
  sem1_wbp_dann_ib               FOUND
  sem1_wbp_caan                  FOUND
  sem1_wl_dann_ib                FOUND
  sem1_wl_caan                   FOUND
  sem1_xlsr_dann_ib              FOUND

Found: 22/22


## Create Dataloaders

In [ ]:
MAX_LEN = 64000
EVAL_BATCH_SIZE = 48
_eval_cfg = GLADAConfig(backbone='wavlm-base-plus')
_eval_cfg.max_audio_len = MAX_LEN

eval_loaders = {}

# --- 2019_LA ---
p = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
a = '/content/data/asvspoof/LA/ASVspoof2019_LA_eval/flac'
if os.path.exists(p) and os.path.isdir(a):
    print('Loading 2019_LA...')
    eval_loaders['2019_LA'] = DataLoader(
        ASVspoofEvalDataset(p, a, _eval_cfg),
        batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
else:
    print(f'2019_LA: MISSING (proto={os.path.exists(p)}, audio={os.path.isdir(a)})')

# --- ITW ---
p = '/content/data/itw/release_in_the_wild/meta.csv'
a = '/content/data/itw/release_in_the_wild'
if os.path.exists(p):
    print('Loading ITW...')
    eval_loaders['ITW'] = DataLoader(
        InTheWildDataset(p, a, _eval_cfg, use_labels=True),
        batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
else:
    print('ITW: MISSING')

# --- 2021_LA (FIXED path: protocol inside eval dir, not keys/) ---
p = '/content/data/asvspoof21_la/ASVspoof2021_LA_eval/ASVspoof2021.LA.cm.eval.trl.txt'
a = '/content/data/asvspoof21_la/ASVspoof2021_LA_eval/flac'
if os.path.exists(p) and os.path.isdir(a):
    print('Loading 2021_LA...')
    eval_loaders['2021_LA'] = DataLoader(
        ASVspoof2021Dataset(p, a, max_len=MAX_LEN),
        batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
else:
    print(f'2021_LA: MISSING (proto={os.path.exists(p)}, audio={os.path.isdir(a)})')

# --- 2021_DF ---
p = '/content/data/asvspoof21_df/keys/DF/CM/trial_metadata.txt'
a = '/content/data/asvspoof21_df/flac'
if os.path.exists(p) and os.path.isdir(a):
    print('Loading 2021_DF...')
    eval_loaders['2021_DF'] = DataLoader(
        ASVspoof2021Dataset(p, a, max_len=MAX_LEN),
        batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
else:
    print(f'2021_DF: MISSING (keys={os.path.exists(p)}, audio={os.path.isdir(a)})')

# --- WaveFake ---
wf = '/content/data/wavefake'
# Find the actual root with audio (may be nested under generated_audio/)
wf_root = wf
if os.path.isdir(os.path.join(wf, 'generated_audio')):
    wf_root = os.path.join(wf, 'generated_audio')
if os.path.isdir(wf_root):
    print('Loading WaveFake...')
    ds = WaveFakeDataset(wf_root, max_len=MAX_LEN)
    if len(ds) > 0 and sum(1 for s in ds.samples if s['label'] == 1) > 0:
        eval_loaders['WaveFake'] = DataLoader(
            ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    else:
        print('  WaveFake: Skipping (no bonafide samples, cannot compute EER)')
else:
    print('WaveFake: MISSING')

print(f'\nLoaders ready: {list(eval_loaders.keys())}')

Loading 2019_LA...
  ASVspoof Eval: 71237 samples
Loading ITW...
  ITW: 31779 samples (labeled returned)
Loading 2021_LA...
  ASVspoof 2021: 0 samples (0 bonafide, 0 spoof)
2021_DF: MISSING (keys=True, audio=False)
Loading WaveFake...
  WaveFake: 134266 samples (0 bonafide, 134266 spoof)
  WaveFake only distributes generated audio.
  Bonafide LJSpeech-1.1 must be downloaded separately.
  WaveFake: Skipping (no bonafide samples, cannot compute EER)

Loaders ready: ['2019_LA', 'ITW', '2021_LA']


## Run Evaluation

In [ ]:
start_time = time.time()
n_total = len(ALL_MODELS)
n_done = 0
n_skipped = 0
n_failed = 0

for i, (name, ckpt_path, loader_type, backbone) in enumerate(ALL_MODELS):
    print(f'\n{"="*70}')
    print(f'[{i+1}/{n_total}] {name}')
    print(f'{"="*70}')

    existing_ds = set()
    if name in all_results:
        ds_data = all_results[name].get('datasets', {})
        existing_ds = {k for k, v in ds_data.items()
                       if isinstance(v, dict) and v.get('eer', -1) >= 0}

    missing_ds = set(eval_loaders.keys()) - existing_ds
    if not missing_ds:
        print(f'  All done. Skipping.')
        n_skipped += 1
        continue
    print(f'  Have: {sorted(existing_ds) or "none"}, Need: {sorted(missing_ds)}')

    if ckpt_path == 'NOT_FOUND' or not os.path.exists(ckpt_path):
        print(f'  Checkpoint not found. SKIPPING.')
        n_failed += 1
        continue

    try:
        if loader_type == 'glada':
            model, _ = load_glada_model(ckpt_path, backbone)
        else:
            model, _ = load_legacy_model(ckpt_path, backbone)
        model = model.to(device)
    except Exception as e:
        print(f'  LOAD ERROR: {e}')
        n_failed += 1
        continue

    new_results = evaluate_missing_only(
        model, eval_loaders, existing_ds, device, name, amp_dtype=AMP_DTYPE)

    if name not in all_results:
        all_results[name] = {
            'checkpoint': ckpt_path, 'loader_type': loader_type,
            'backbone': backbone, 'datasets': {},
            'timestamp': datetime.datetime.now().isoformat(),
        }
    all_results[name]['datasets'].update(new_results)
    all_results[name]['timestamp'] = datetime.datetime.now().isoformat()

    with open(RESULTS_FILE, 'w') as f:
        json.dump(all_results, f, indent=2, default=str)

    del model
    torch.cuda.empty_cache()
    n_done += 1

    elapsed = time.time() - start_time
    per_model = elapsed / max(n_done, 1)
    remaining = per_model * (n_total - i - 1)
    print(f'  Elapsed: {elapsed/60:.1f}m, Remaining: ~{remaining/60:.1f}m')

print(f'\n{"="*70}')
print(f'DONE: {n_done} evaluated, {n_skipped} skipped, {n_failed} failed')
print(f'Results: {RESULTS_FILE}')


[1/22] full_glada
  Have: ['2019_LA', 'ITW'], Need: ['2021_LA']


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

  GLADA: wavlm-base-plus, ep=3, EER=19.636561007871332
  2019_LA        : SKIP (already done)
  ITW            : SKIP (already done)


... (88 outputs left out)

  LOAD ERROR: Error(s) in loading state_dict for LegacyDomainAdaptationModel:
	size mismatch for classifier.1.weight: copying a param with shape torch.Size([2, 256]) from checkpoint, the shape in current model is torch.Size([256]).
	size mismatch for classifier.1.bias: copying a param with shape torch.Size([2]) from checkpoint, the shape in current model is torch.Size([256]).
	size mismatch for domain_disc.0.weight: copying a param with shape torch.Size([64, 25]) from checkpoint, the shape in current model is torch.Size([256, 16]).
	size mismatch for domain_disc.0.bias: copying a param with shape torch.Size([64]) from checkpoint, the shape in current model is torch.Size([256]).

[22/22] sem1_xlsr_dann_ib
  Have: none, Need: ['2019_LA', '2021_LA', 'ITW']


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  LOAD ERROR: Error(s) in loading state_dict for LegacyDomainAdaptationModel:
	size mismatch for classifier.1.weight: copying a param with shape torch.Size([2, 256]) from checkpoint, the shape in current model is torch.Size([256]).
	size mismatch for classifier.1.bias: copying a param with shape torch.Size([2]) from checkpoint, the shape in current model is torch.Size([256]).
	size mismatch for domain_disc.0.weight: copying a param with shape torch.Size([64, 24]) from checkpoint, the shape in current model is torch.Size([256, 16]).
	size mismatch for domain_disc.0.bias: copying a param with shape torch.Size([64]) from checkpoint, the shape in current model is torch.Size([256]).

DONE: 16 evaluated, 0 skipped, 6 failed
Results: /content/drive/MyDrive/Jakob/Experiments/comprehensive_eval/results.json


## Results Summary

In [ ]:
with open(RESULTS_FILE) as f:
    all_results = json.load(f)

DS_ORDER = ['2019_LA', 'ITW', '2021_LA', '2021_DF', 'WaveFake']

header = '| Model | ' + ' | '.join(DS_ORDER) + ' |'
sep = '|---|' + '---|' * len(DS_ORDER)
print(header)
print(sep)
for name in [m[0] for m in ALL_MODELS]:
    if name not in all_results:
        continue
    row = f'| {name} |'
    ds = all_results[name].get('datasets', {})
    for d in DS_ORDER:
        r = ds.get(d, {})
        if isinstance(r, dict) and r.get('eer', -1) >= 0:
            row += f' {r["eer"]:.2f} |'
        elif isinstance(r, dict) and 'error' in r:
            row += ' ERR |'
        else:
            row += ' - |'
    print(row)

total = sum(1 for n in all_results for d in DS_ORDER
            if isinstance(all_results[n].get('datasets',{}).get(d,{}), dict)
            and all_results[n]['datasets'][d].get('eer', -1) >= 0)
print(f'\nCoverage: {total}/{len(all_results)*len(DS_ORDER)} pairs')

| Model | 2019_LA | ITW | 2021_LA | 2021_DF | WaveFake |
|---|---|---|---|---|---|
| full_glada | 1.04 | 19.71 | 50.00 | - | - |
| no_glada | 1.36 | 14.89 | 50.00 | - | - |
| no_vib | 0.72 | 14.58 | 50.00 | - | - |
| no_speaker | 0.93 | 13.48 | 50.00 | - | - |
| no_cka | 1.07 | 12.98 | 50.00 | - | - |
| no_rawboost | 1.05 | 16.66 | 50.00 | - | - |
| no_hier | 1.15 | 17.47 | 50.00 | - | - |
| global_da_only | 1.42 | 14.99 | 50.00 | - | - |
| hierarchical_only | 1.02 | 13.28 | 50.00 | - | - |
| no_da | 0.80 | 14.20 | 50.00 | - | - |
| backbone_only | 0.62 | 13.60 | 50.00 | - | - |
| bb_wavlm-large_s42 | 4.08 | 12.64 | 50.00 | - | - |
| bb_wavlm-large_s123 | 1.28 | 15.93 | 50.00 | - | - |
| bb_xlsr-300m_s42 | 0.71 | 18.40 | 50.00 | - | - |
| bb_data2vec-large | 1.68 | 18.14 | 50.00 | - | - |
| bb_wav2vec2-robust | ERR | 21.98 | 50.00 | - | - |


KeyError: '2021_DF'

In [ ]:
print(f'Results: {RESULTS_FILE}')
try:
    from google.colab import files
    files.download(RESULTS_FILE)
except:
    print('Download from Drive: Jakob/Experiments/comprehensive_eval/results.json')